# Advanced Machine Learning - Final Project
## Anti-Money Laundering (AML) Transaction Classification

**Authors:** Giovanni Pacchetti, Asier Larrazabal and Asier Aurre

---

### Project Overview
The detection of money laundering is a classic example of an **extreme anomaly detection problem**. Illicit transactions represent a tiny fraction of overall banking volume, meaning our models will face severe class imbalance. 

To tackle this systematically, we have divided our project into three distinct notebooks:
1. **Notebook 1: Data Acquisition & Initial Inspection** - Fetching the raw datasets and parsing irregular file formats.
2. **Notebook 2: Data Integration** - Merging the raw transactions with the identified laundering patterns into a unified dataset.
3. **Notebook 3: Classification & Machine Learning** - Applying preprocessing, advanced resampling (Undersampling + SMOTE) and evaluating Ensemble models.

### 1. Data Acquisition
To ensure our code is reproducible regardless of the machine it runs on, we download our raw datasets directly from Google Drive using `gdown`.

In [ ]:
# Import necessary libraries for file management and downloading
import gdown
import os

# Define the target directory to keep our workspace clean
folder_name = 'data'

# We use a dictionary to map Google Drive File IDs to their target filenames.
# This makes it easy to scale if we need to add more datasets later.
files_to_download = {
    '1EIvVyArDU3MfAohIo5-AsUwmQO_ywowH': 'ibm_aml_multiclass_clases.csv', # Pre-merged reference dataset
    '10g5-Cuh2unWxQnbm9Wq1eCzhhmMFlsmy': 'HI-Medium_Trans.csv',           # Raw transactions
    '1qFEGELsSobpcuvPmdOJ8JK0GIqtWqSp3': 'HI-Medium_Patterns.txt'         # Identified laundering patterns
}

# Create the data directory if it doesn't exist yet
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"Folder '{folder_name}' created.\n")

# Iterate through the dictionary and download each file
for file_id, file_name in files_to_download.items():
    output_path = os.path.join(folder_name, file_name)
    url = f'https://drive.google.com/uc?id={file_id}'
    
    print(f"Downloading: {file_name}...")
    # gdown automatically checks if the file exists to avoid redundant downloads
    gdown.download(url, output_path, quiet=False)
    print(f"Saved at: {output_path}\n")

print("All downloads completed successfully!")

Downloading: ibm_aml_multiclass_clases.csv...


Downloading...
From (original): https://drive.google.com/uc?id=1EIvVyArDU3MfAohIo5-AsUwmQO_ywowH
From (redirected): https://drive.google.com/uc?id=1EIvVyArDU3MfAohIo5-AsUwmQO_ywowH&confirm=t&uuid=21485fb0-69d3-45b6-830c-708cfbc8dacf
To: c:\Users\chivi\Desktop\uni\4\AdvancedMachineLearning\MoneyLaunderingClassifier\data\ibm_aml_multiclass_clases.csv
100%|██████████| 3.07G/3.07G [02:51<00:00, 17.9MB/s]


Saved at: data\ibm_aml_multiclass_clases.csv

Downloading: HI-Medium_Trans.csv...


Downloading...
From (original): https://drive.google.com/uc?id=10g5-Cuh2unWxQnbm9Wq1eCzhhmMFlsmy
From (redirected): https://drive.google.com/uc?id=10g5-Cuh2unWxQnbm9Wq1eCzhhmMFlsmy&confirm=t&uuid=7aa92b3e-00a4-497a-bed3-5c5829ca9e04
To: c:\Users\chivi\Desktop\uni\4\AdvancedMachineLearning\MoneyLaunderingClassifier\data\HI-Medium_Trans.csv
100%|██████████| 3.03G/3.03G [03:04<00:00, 16.4MB/s]


Saved at: data\HI-Medium_Trans.csv

Downloading: HI-Medium_Patterns.txt...


Downloading...
From: https://drive.google.com/uc?id=1qFEGELsSobpcuvPmdOJ8JK0GIqtWqSp3
To: c:\Users\chivi\Desktop\uni\4\AdvancedMachineLearning\MoneyLaunderingClassifier\data\HI-Medium_Patterns.txt
100%|██████████| 2.31M/2.31M [00:00<00:00, 3.81MB/s]

Saved at: data\HI-Medium_Patterns.txt

All downloads completed!


### 2. Data Loading and Initial Parsing
With the files downloaded, we can load them into memory. 

The transaction dataset (`HI-Medium_Trans.csv`) is a standard CSV and loads natively. However, the patterns dataset (`HI-Medium_Patterns.txt`) requires custom parsing. It is a text file that may contain irregular lines or missing headers. We use Python's `StringIO` and list comprehensions to filter out invalid rows before feeding the clean string data into a Pandas DataFrame.

In [ ]:
import pandas as pd
from pathlib import Path
from io import StringIO

# --- 2.1 Loading Raw Transactions ---
DATA_PATH = Path("data/HI-Medium_Trans.csv")  

print("Loading primary transactions dataset...")
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows and {df.shape[1]} columns")
print("\n--- First 5 rows of Transactions ---")
display(df.head())

# --- 2.2 Parsing and Loading Patterns ---
PATTERNS_PATH = Path("data/HI-Medium_Patterns.txt")
print("\nLoading and parsing patterns text file...")

# Read the raw text file
patterns_txt = PATTERNS_PATH.read_text()

# Filter out malformed lines (only keep lines that contain commas)
# This prevents parsing errors when creating the DataFrame
lines = [line for line in patterns_txt.split('\n') if ',' in line]

# Use StringIO to simulate a file object for pandas, passing our cleaned lines
patterns_df = pd.read_csv(StringIO('\n'.join(lines)), header=None)

# Manually assign the correct schema
patterns_df.columns = [
    'timestamp', 'source_id', 'source_account', 'destination_id', 'destination_account',
    'amount', 'currency_out', 'amount_in', 'currency_in', 'method', 'flag'
]

print("Patterns loaded successfully!")
print(f"Shape: {patterns_df.shape[0]:,} rows and {patterns_df.shape[1]} columns")
print("\n--- First 5 rows of Patterns ---")
display(patterns_df.head())

Loading primary transactions dataset...
Dataset loaded successfully!
Shape: 31,898,238 rows and 11 columns

--- First 5 rows of Transactions ---


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,0
1,2022/09/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0
2,2022/09/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,0
3,2022/09/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,0
4,2022/09/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,0



Loading and parsing patterns text file...
Patterns loaded successfully!
Shape: 22,743 rows and 11 columns

--- First 5 rows of Patterns ---


,timestamp,source_id,source_account,destination_id,destination_account,amount,currency_out,amount_in,currency_in,method,flag
0,2022/09/01 05:14,952,8139F54E0,111632,8062C56E0,5331.44,US Dollar,5331.44,US Dollar,ACH,1
1,2022/09/03 13:09,111632,8062C56E0,8456,81363F620,5602.59,US Dollar,5602.59,US Dollar,ACH,1
2,2022/09/01 07:40,118693,823D5EB90,13729,801CF2E60,1400.54,US Dollar,1400.54,US Dollar,ACH,1
3,2022/09/01 14:19,13729,801CF2E60,123621,81A7090F0,1467.94,US Dollar,1467.94,US Dollar,ACH,1
4,2022/09/02 12:40,24750,81363F410,213834,808757B00,16898.29,US Dollar,16898.29,US Dollar,ACH,1
